# Enterprise Text-to-SQL — Self-Correction Repair Loop

**Author:** Garvit Audichya ([GitHub](https://github.com/garvit-010/Voice-Text-to-SQL-FineTuning))

This notebook executes the model-based repair loop on queries that failed static validation or database execution. Demonstrates how single-turn self-correction lifts accuracy from **68.43% to 70.86%** on held-out enterprise queries.


In [ ]:
!pip install -q transformers>=5.0 peft>=0.20 bitsandbytes>=0.50 sqlglot huggingface-hub
print('Setup complete.')

In [ ]:
import os, json, time, re
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ADAPTER_ID = 'garvit-010/qwen3-8b-text2sql-qlora-v3'
BASE_MODEL = 'Qwen/Qwen3-8B'

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision='b968826d9c46',
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    ),
    device_map={'': 0},
)
model = PeftModel.from_pretrained(base, ADAPTER_ID).eval()
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
print('Repair model loaded.')
